<a href="https://colab.research.google.com/github/vladislava-c/process-automation-analyzer/blob/main/Process_to_automation_ideas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openai -q

In [60]:
# Each step = (step text, your label, your preferred Microsoft tool or "-")
TOOLS = ["Power Automate", "Power Apps","Copilot", "SharePoint", "Outlook", "Teams", "Microsoft Forms", "Power BI"]

LABELS = ["AUTOMATE", "ASSIST", "MANUAL"]

ANSWER_KEY = [
 {"id": 16, "process": "A customer applies for a small business loan online. We check the documents, an analyst writes a recommendation, the credit committee decides, and the customer is told the outcome.",
  "steps": [
   ("Collect loan application and documents online", "AUTOMATE", "Microsoft Forms"),   # borderline: Power Apps
   ("Check that all required documents are present", "AUTOMATE", "Power Automate"),
   ("Draft credit recommendation for the analyst", "ASSIST", "Copilot"),
   ("Credit committee decides on the loan", "MANUAL", "-"),
   ("Inform customer of the decision", "AUTOMATE", "Outlook")]},                        # borderline: rejections may need ASSIST

 {"id": 17, "process": "Every quarter the system owners must confirm who still needs access. Someone exports the user list, sends it to each owner, the owners confirm who can stay, unusual access is looked at by security, and unconfirmed accounts are removed. Completion rates are reported.",
  "steps": [
   ("Export current user access list", "AUTOMATE", "Power Automate"),
   ("Send list to each system owner for review", "AUTOMATE", "Outlook"),                # borderline: Power Automate or Teams
   ("Owner confirms who keeps access", "MANUAL", "-"),
   ("Flag unusual or high-risk access", "ASSIST", "Copilot"),
   ("Remove accounts that were not confirmed", "AUTOMATE", "Power Automate"),          # borderline: MANUAL in many companies
   ("Report review completion rates", "AUTOMATE", "Power BI")]},

 {"id": 18, "process": "A customer asks for a copy of their personal data. Compliance verifies who they are, collects the data from the different systems, checks that it contains nobody else's information, approves the release, and makes sure the 30-day deadline is met.",
  "steps": [
   ("Receive and log the request", "AUTOMATE", "Microsoft Forms"),                    # borderline: Outlook
   ("Verify the identity of the requester", "MANUAL", "-"),
   ("Collect data from the different systems", "MANUAL", "-"),                          # borderline: AUTOMATE if connectors exist
   ("Check the data for other people's information", "ASSIST", "Copilot"),
   ("Approve release of the data package", "MANUAL", "-"),
   ("Track the 30-day deadline", "AUTOMATE", "Power Automate")]},                      # borderline: SharePoint

 {"id": 19, "process": "so basically when someone from sales wants a discount over 10% they message me, i look at the customer history, ask my boss if its ok, then tell sales yes or no and put it in the excel",
  "steps": [
   ("Receive discount request from sales", "AUTOMATE", "Microsoft Forms"),             # borderline: Teams
   ("Review the customer's history", "ASSIST", "Copilot"),
   ("Route the request to the manager for approval", "AUTOMATE", "Power Automate"),
   ("Manager decides on the discount", "MANUAL", "-"),
   ("Tell sales the decision", "AUTOMATE", "Teams"),
   ("Record the decision in the spreadsheet", "AUTOMATE", "SharePoint")]},             # borderline: Power Automate

 {"id": 20, "process": "we get supplier emails all the time with price changes. somebody reads them, updates our price list, and if its a big jump they call the supplier. once a month the buyers meet and talk about which suppliers to push on price",
  "steps": [
   ("Read supplier emails about price changes", "ASSIST", "Copilot"),
   ("Update the price list", "AUTOMATE", "Power Automate"),                             # borderline: SharePoint
   ("Flag price increases above a threshold", "AUTOMATE", "Power Automate"),
   ("Call the supplier about large increases", "MANUAL", "-"),
   ("Summarise price trends for the monthly buyer meeting", "ASSIST", "Copilot"),       # borderline: Power BI
   ("Decide which suppliers to push on price", "MANUAL", "-")]},
]
print("Loaded", len(ANSWER_KEY), "processes")

Loaded 5 processes


In [61]:
from google.colab import userdata
from openai import OpenAI
import json
import pandas as pd
import gradio as gr

client = OpenAI(api_key=userdata.get("API"))
MODEL = "gpt-4o-mini"

EVAL_INSTRUCTIONS = f"""
You analyze business processes for a company that uses the Microsoft landscape.
You get a process and a numbered list of steps. For each step assign exactly one label:
- AUTOMATE: rule-based, repeatable, structured data, no judgment needed
- ASSIST: AI or software can prepare or draft, but a human must review or approve
- MANUAL: needs human judgment, relationships, accountability, or legal sign-off
Also give the best-fit Microsoft tool, using EXACTLY one name from this list: {TOOLS}.
For MANUAL steps, the tool is "-".
Return JSON only: {{"steps": [{{"n": 1, "label": "AUTOMATE|ASSIST|MANUAL", "tool": "..."}}]}}
One entry per step, in the same order.

Rules for choosing tools:
- Power Automate cannot read or extract data from the content of documents (forms, receipts, invoices, CVs, contracts). Never choose it for that.
- In our setup, extracting content from documents is done by a person: label that step MANUAL with tool "-".
- This applies ONLY to the step that reads the content. Steps that collect, route, store, send or check the completeness of documents are still AUTOMATE with Power Automate.
Tool selection rules. Use these exact names only: Power Automate, Power Apps, Copilot, SharePoint, Outlook, Teams, Microsoft Forms, Power BI.
- Sending or scheduling email or calendar items: Outlook
- Notifying or reminding people in chat: Teams
- Collecting submissions or survey responses: Microsoft Forms
- Building a custom input screen or app for users: Power Apps
- Routing, approvals, data transfer, triggers, and other system-to-system steps: Power Automate
- Storing, tracking, or publishing documents and lists: SharePoint
- Dashboards and reports: Power BI
- Summarising, drafting, classifying, comparing, or extracting from text: Copilot
- Always pick up the less complex tool to use for automation
ASSIST means a step where AI or software prepares something (a draft, a summary, a classification, a proposal) and a human reviews or approves it.
Use ASSIST, not MANUAL, whenever AI could do the first draft of the work.
Use MANUAL only for decisions, negotiation, investigation, accountability, or sign-off that a person must own.
"""

def label_steps(process_text, steps):
    numbered = "\n".join(f"{i+1}. {s[0]}" for i, s in enumerate(steps))
    user_msg = f"Process: {process_text}\n\nSteps:\n{numbered}"
    r = client.chat.completions.create(
        model=MODEL,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": EVAL_INSTRUCTIONS},
                  {"role": "user", "content": user_msg}],
    )
    return json.loads(r.choices[0].message.content)["steps"]

rows = []
for p in ANSWER_KEY:
    try:
        out = label_steps(p["process"], p["steps"])
    except Exception as e:
        print("Error on", p["id"], e)
        continue
    for i, (step, key_label, key_tool) in enumerate(p["steps"]):
        m = out[i] if i < len(out) else {"label": "?", "tool": "?"}
        rows.append({"id": p["id"], "step": step,
                     "key_label": key_label, "model_label": m.get("label", "?"),
                     "key_tool": key_tool, "model_tool": m.get("tool", "?")})

df = pd.DataFrame(rows)
df["label_ok"] = df["key_label"] == df["model_label"]
df["tool_ok"] = df["key_tool"].str.lower() == df["model_tool"].str.lower()

tool_rows = df[df["key_tool"] != "-"]
print(f"Label accuracy: {df['label_ok'].mean():.0%}  ({df['label_ok'].sum()}/{len(df)} steps)")
print(f"Tool accuracy:  {tool_rows['tool_ok'].mean():.0%}  ({tool_rows['tool_ok'].sum()}/{len(tool_rows)} automatable steps)")
print()
print(pd.crosstab(df["key_label"], df["model_label"], rownames=["Your key"], colnames=["Model said"]))

Label accuracy: 83%  (24/29 steps)
Tool accuracy:  57%  (12/21 automatable steps)

Model said  ASSIST  AUTOMATE  MANUAL
Your key                            
ASSIST           2         0       4
AUTOMATE         0        14       1
MANUAL           0         0       8


In [59]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 220)
df[~df["label_ok"] | (~df["tool_ok"] & (df["key_tool"] != "-"))][
    ["id", "step", "key_label", "model_label", "key_tool", "model_tool"]]

,id,step,key_label,model_label,key_tool,model_tool
6,12,Collect responses,AUTOMATE,AUTOMATE,Microsoft Forms,Power Automate
10,13,Collect incident report,AUTOMATE,AUTOMATE,Power Apps,Microsoft Forms
11,13,Log incident in central list,AUTOMATE,AUTOMATE,SharePoint,Power Automate
14,13,Alert management for serious incidents,AUTOMATE,AUTOMATE,Teams,Power Automate
15,14,Track contract end dates,AUTOMATE,AUTOMATE,SharePoint,Power Automate
16,14,Remind contract owner 90 days before expiry,AUTOMATE,AUTOMATE,Outlook,Power Automate
19,14,File signed contract,AUTOMATE,AUTOMATE,SharePoint,Power Automate
20,15,Collect weekly updates from team members,AUTOMATE,AUTOMATE,Microsoft Forms,Power Automate
